In [ ]:
"""
Step 3c - Feature Engineering (Silver -> customer-month features)
Customer Health & Sentiment Scoring Multi-Agent AI Solution

Grain: ONE row per customer_id + month (YYYY-MM), Oct-2025 .. Sep-2026.
Runs on the FULL Silver population - the POC cohort is applied afterwards in Step 3d, so
population-relative features (spend percentile) and rolling history are never distorted.

How it works
  0. Verify the business mappings (business_rules.py) against Silver values - STRICT by default.
  1. Aggregate transactions, engagements and interactions SEPARATELY by customer_id + month
     (avoids event-level row multiplication).
  2. Build a dense customer x month spine (from signup month or window start), so that
     "no activity" months exist and rolling windows / recency are correct.
  3. Left-join the three monthly aggregates (counts zero-filled AFTER the has_<source>_m flags
     are set, so "no record" stays distinguishable from "zero activity"; averages left NULL).
  4. Add rolling 3-month features, trends, ratios, recency, lifecycle and data-coverage features.
  5. Add profile attributes as CONTEXT ONLY (ctx_ prefix) - not health/sentiment penalties.
  6. Run feature QA and write
        data/features/customer_month_features.parquet   (model inputs - no labels)
        data/features/customer_month_labels.parquet     (sentiment ground truth - evaluation only)

Never used as features (label protection):  sentiment_label, sentiment_score_gt.
interaction_text is handled by the Sentiment Agent, not here.

Run:  python step3c_feature_engineering.py            (strict mapping check)
      python step3c_feature_engineering.py --lenient  (warn only on mapping mismatches)

Change log (see decisions.md): D19 spine no longer drops events dated before signup_date (pre_signup_m flag),
D04 fee features (EDA v5) + fee types optional in the mapping check, D11 rolling
3-month windows driven by business_rules.CORE_LOOKBACK_MONTHS (was hard-coded), D09 notebook-safe finish().
"""

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime

import business_rules as br
from business_rules import isin
from common import (FEATURE_DIR, LABEL_COLUMNS, REPORT_DIR, SILVER_DIR, WINDOW_END,
                    WINDOW_START, copy_to_csv, copy_to_parquet, get_connection, log,
                    register_layer, finish, scalar)

OUT = REPORT_DIR / "features"
FEATURE_FILE = FEATURE_DIR / "customer_month_features.parquet"
LABEL_FILE = FEATURE_DIR / "customer_month_labels.parquet"

# Amounts are aggregated in their own currency. Fill this ONLY with dated, approved rates;
# until then spend is compared through the within-(month, currency) percentile.
FX_TO_BASE: dict[str, float] = {}   # e.g. {"GBP": 1.0, "EUR": 0.85}

ELIGIBILITY_LOOKBACK_MONTHS = br.CORE_LOOKBACK_MONTHS   # >= 1 event in any source in trailing N months
NUMERIC_TYPES = ("TINYINT", "SMALLINT", "INTEGER", "BIGINT", "HUGEINT", "FLOAT", "DOUBLE", "DECIMAL")

## 2. Currency handling

In [ ]:
def fx_expr() -> str:
    if not FX_TO_BASE:
        return "amount"
    cases = " ".join(f"WHEN '{k}' THEN {v}" for k, v in FX_TO_BASE.items())
    return f"amount * CASE currency {cases} ELSE NULL END"

## 3. Mapping check (business_rules vs Silver values)

In [ ]:
def check_mappings(con, strict: bool) -> list[str]:
    """Configured values must exist in Silver, and every engagement event_type must be classified."""
    checks = [
        ("transactions", "transaction_type",
         br.TX_SPEND_TYPES + br.TX_PAYMENT_TYPES + br.TX_REFUND_TYPES + br.TX_REFUND_BASE_TYPES, False),
        ("transactions", "transaction_status",
         br.TX_SUCCESS_STATUSES + br.TX_FAILED_STATUSES + br.TX_REVERSED_STATUSES, False),
        ("engagements", "event_type", br.ENG_POSITIVE + br.ENG_NEUTRAL + br.ENG_NEGATIVE, True),
        ("interactions", "interaction_type", br.INT_COMPLAINT_TYPES, False),
        ("interactions", "resolution_status", br.INT_OPEN_STATUSES + br.INT_RESOLVED_STATUSES, False),
    ]
    issues: list[str] = []
    log("Checking business mappings (business_rules.py) against Silver values")
    for t, c, configured, must_cover_all in checks:
        present = {r[0] for r in con.execute(f"SELECT DISTINCT {c} FROM silver_{t} WHERE {c} IS NOT NULL").fetchall()}
        missing = sorted(set(configured) - present)
        unmapped = sorted(present - set(configured))
        log(f"  {t}.{c}: configured-but-absent {missing or '-'} | present-but-unmapped {unmapped or '-'}")
        issues += [f"{t}.{c}: configured value '{v}' not found in data" for v in missing]
        if must_cover_all:
            issues += [f"{t}.{c}: value '{v}' is not classified" for v in unmapped]
    multi_ccy = con.execute("""SELECT count(*) FROM (SELECT customer_id FROM silver_transactions
                               GROUP BY 1 HAVING count(DISTINCT currency) > 1)""").fetchall()[0][0]
    if multi_ccy and not FX_TO_BASE:
        log(f"  NOTE: {multi_ccy:,} customers transact in >1 currency - amounts stay native; "
            "use spend_pctile_m for cross-customer comparison.")
    if br.TX_FEE_TYPES:   # optional group: absence is a NOTE, never a mapping failure
        present = {r[0] for r in con.execute("SELECT DISTINCT transaction_type FROM silver_transactions").fetchall()}
        absent = sorted(set(br.TX_FEE_TYPES) - present)
        if absent:
            log(f"  NOTE: fee type(s) {absent} not found - fee features will be 0 (set TX_FEE_TYPES = () to silence)")
    for i in issues:
        log(f"  MAPPING ISSUE: {i}")
    if issues and strict:
        raise ValueError(f"{len(issues)} mapping issue(s) - fix business_rules.py or run with --lenient")
    return issues

## 4. Monthly aggregates per source

In [ ]:
def monthly_transactions(con) -> None:
    # Classification uses type + status only. Amount sign is a Step 3b WARN, never a classifier,
    # so a negative correction can no longer be counted as a refund AND a reversal.
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE m_txn AS
        WITH t AS (
            SELECT *, {fx_expr()} AS amount_base,
                   {isin('transaction_type', br.TX_SPEND_TYPES)}         AS is_spend_type,
                   {isin('transaction_type', br.TX_PAYMENT_TYPES)}       AS is_payment_type,
                   {isin('transaction_type', br.TX_REFUND_TYPES)}        AS is_refund_type,
                   {isin('transaction_type', br.TX_FEE_TYPES)}           AS is_fee_type,
                   {isin('transaction_status', br.TX_SUCCESS_STATUSES)}  AS is_success,
                   {isin('transaction_status', br.TX_FAILED_STATUSES)}   AS is_failed,
                   {isin('transaction_status', br.TX_REVERSED_STATUSES)} AS is_reversed
            FROM silver_transactions
        )
        SELECT customer_id, month,
               count(*)                                                                AS txn_count_m,
               count(*) FILTER (WHERE is_success)                                      AS txn_success_count_m,
               count(*) FILTER (WHERE is_failed)                                       AS txn_failed_count_m,
               count(*) FILTER (WHERE is_spend_type AND is_success)                    AS purchase_count_m,
               COALESCE(sum(amount_base) FILTER (WHERE is_spend_type AND is_success), 0) AS purchase_amount_m,
               count(*) FILTER (WHERE is_payment_type)                                 AS payment_attempts_m,
               count(*) FILTER (WHERE is_payment_type AND is_failed)                   AS failed_payment_count_m,
               count(*) FILTER (WHERE is_refund_type AND is_success)                   AS refund_count_m,
               COALESCE(sum(abs(amount_base)) FILTER (WHERE is_refund_type AND is_success), 0) AS refund_amount_m,
               count(*) FILTER (WHERE is_reversed)                                     AS reversal_count_m,
               count(*) FILTER (WHERE is_fee_type AND is_success)                      AS fee_count_m,
               COALESCE(sum(amount_base) FILTER (WHERE is_fee_type AND is_success), 0) AS fee_amount_m,
               COALESCE(sum(abs(amount_base)) FILTER (WHERE is_fee_type AND is_reversed), 0) AS fee_reversed_amount_m,
               COALESCE(sum(discount_amount) FILTER (WHERE is_spend_type AND is_success), 0) AS discount_amount_m,
               count(*) FILTER (WHERE is_international)                                AS intl_txn_count_m,
               count(DISTINCT product_category) FILTER (WHERE is_spend_type AND is_success) AS distinct_categories_m,
               count(DISTINCT currency)                                                AS currency_count_m,
               mode(currency)                                                          AS txn_currency_m,
               max(transaction_ts) FILTER (WHERE is_spend_type AND is_success)         AS last_purchase_ts_m
        FROM t GROUP BY ALL
    """)


def monthly_engagements(con) -> None:
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE m_eng AS
        SELECT customer_id, month,
               count(*)                                                            AS eng_event_count_m,
               count(DISTINCT CAST(event_ts AS DATE))                              AS active_days_m,
               count(*) FILTER (WHERE {isin('event_type', br.ENG_LOGIN)})          AS login_count_m,
               count(*) FILTER (WHERE {isin('event_type', br.ENG_POSITIVE)})       AS eng_positive_count_m,
               count(*) FILTER (WHERE {isin('event_type', br.ENG_NEUTRAL)})        AS eng_neutral_count_m,
               count(*) FILTER (WHERE {isin('event_type', br.ENG_NEGATIVE)})       AS eng_negative_count_m,
               COALESCE(sum(session_duration_sec), 0) / 60.0                       AS session_minutes_m,
               COALESCE(sum(pages_viewed), 0)                                      AS pages_viewed_m,
               count(*) FILTER (WHERE is_converted)                                AS conversion_count_m,
               count(*) FILTER (WHERE {isin('event_type', br.ENG_OPT_OUT)})        AS opt_out_count_m,
               count(*) FILTER (WHERE {isin('event_type', br.ENG_DISMISS)})        AS dismiss_count_m,
               count(*) FILTER (WHERE {isin('event_type', br.ENG_EMAIL_OPEN)})     AS email_open_count_m,
               count(*) FILTER (WHERE {isin('event_type', br.ENG_EMAIL_CLICK)})    AS email_click_count_m,
               count(*) FILTER (WHERE campaign_id IS NOT NULL)                     AS campaign_touch_count_m,
               count(DISTINCT engagement_channel)                                  AS eng_channel_count_m,
               max(event_ts)                                                       AS last_engagement_ts_m
        FROM silver_engagements GROUP BY ALL
    """)


def monthly_interactions(con) -> None:
    # sentiment_label / sentiment_score_gt / interaction_text are deliberately NOT read here
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE m_int AS
        WITH i AS (
            SELECT customer_id, month, interaction_ts, interaction_type, resolution_status,
                   first_contact_resolution, csat_score, nps_score, duration_sec, channel,
                   date_diff('day', lag(interaction_ts) OVER (PARTITION BY customer_id
                                                              ORDER BY interaction_ts),
                             interaction_ts) <= {br.REPEAT_CONTACT_DAYS}     AS is_repeat
            FROM silver_interactions
        )
        SELECT customer_id, month,
               count(*)                                                                         AS interaction_count_m,
               count(*) FILTER (WHERE {isin('interaction_type', br.INT_COMPLAINT_TYPES)})       AS complaint_count_m,
               count(*) FILTER (WHERE {isin('interaction_type', br.INT_BILLING_DISPUTE_TYPES)}) AS billing_dispute_count_m,
               count(*) FILTER (WHERE {isin('interaction_type', br.INT_CANCELLATION_TYPES)})    AS cancellation_request_count_m,
               count(*) FILTER (WHERE {isin('resolution_status', br.INT_OPEN_STATUSES)})        AS open_issue_count_m,
               count(*) FILTER (WHERE {isin('resolution_status', br.INT_ESCALATED_STATUSES)})   AS escalated_count_m,
               count(*) FILTER (WHERE is_repeat)                                                AS repeat_contact_count_m,
               count(*) FILTER (WHERE first_contact_resolution)                                 AS fcr_yes_m,
               count(first_contact_resolution)                                                  AS fcr_known_m,
               COALESCE(sum(csat_score), 0)                                                     AS csat_sum_m,
               count(csat_score)                                                                AS csat_n_m,
               count(*) FILTER (WHERE csat_score <= 2)                                          AS csat_low_m,
               count(*) FILTER (WHERE csat_score >= 4)                                          AS csat_high_m,
               COALESCE(sum(nps_score), 0)                                                      AS nps_sum_m,
               count(nps_score)                                                                 AS nps_n_m,
               count(*) FILTER (WHERE nps_score <= {br.NPS_DETRACTOR_MAX})                      AS nps_detractor_m,
               count(*) FILTER (WHERE nps_score >  {br.NPS_DETRACTOR_MAX}
                                  AND nps_score <= {br.NPS_PASSIVE_MAX})                        AS nps_passive_m,
               count(*) FILTER (WHERE nps_score >  {br.NPS_PASSIVE_MAX})                        AS nps_promoter_m,
               COALESCE(sum(duration_sec), 0) / 60.0                                            AS handle_minutes_m,
               count(DISTINCT channel)                                                          AS int_channel_count_m,
               max(interaction_ts)                                                              AS last_interaction_ts_m
        FROM i GROUP BY ALL
    """)


def monthly_labels(con) -> None:
    """Sentiment ground truth per customer-month -> separate file, evaluation only."""
    con.execute("""
        CREATE OR REPLACE TEMP TABLE m_lbl AS
        SELECT customer_id, month,
               count(sentiment_label)                                  AS gt_labelled_count_m,
               count(*) FILTER (WHERE sentiment_label = 'positive')    AS gt_positive_count_m,
               count(*) FILTER (WHERE sentiment_label = 'neutral')     AS gt_neutral_count_m,
               count(*) FILTER (WHERE sentiment_label = 'negative')    AS gt_negative_count_m,
               avg(sentiment_score_gt)                                 AS gt_sentiment_score_avg_m
        FROM silver_interactions GROUP BY ALL
    """)

## 5. Dense customer-month spine, join and rolling features

In [ ]:
def numeric_cols(con, table: str) -> list[str]:
    return [r[0] for r in con.execute(f"DESCRIBE {table}").fetchall()
            if r[0] not in ("customer_id", "month") and r[1].startswith(NUMERIC_TYPES)]


def build_customer_month(con) -> None:
    m_cols = {t: numeric_cols(con, t) for t in ("m_txn", "m_eng", "m_int")}
    zero_fill = ",\n".join(f"COALESCE({a}.{c}, 0) AS {c}"
                           for a, t in (("t", "m_txn"), ("e", "m_eng"), ("i", "m_int"))
                           for c in m_cols[t])

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE cm_base AS
        WITH months AS (
            SELECT strftime(d, '%Y-%m') AS month, CAST(d AS DATE) AS month_start,
                   CAST(d + INTERVAL 1 MONTH - INTERVAL 1 DAY AS DATE) AS month_end
            FROM range(DATE '{WINDOW_START}', DATE '{WINDOW_END}' + INTERVAL 1 DAY, INTERVAL 1 MONTH) r(d)
        ),
        active AS (SELECT customer_id, min(month) AS first_event_month
                   FROM (SELECT customer_id, month FROM m_txn UNION ALL SELECT customer_id, month FROM m_eng
                         UNION ALL SELECT customer_id, month FROM m_int) GROUP BY 1),
        spine AS (   -- D19: starts at the EARLIER of signup month and first event month (no event is dropped)
            SELECT c.customer_id, m.month, m.month_start, m.month_end,
                   (c.signup_date IS NOT NULL AND c.signup_date > m.month_end)::INT AS pre_signup_m
            FROM silver_customers c
            JOIN active a USING (customer_id)
            CROSS JOIN months m
            WHERE c.signup_date IS NULL OR c.signup_date <= m.month_end OR m.month >= a.first_event_month
        )
        SELECT s.*,
               {zero_fill},
               t.txn_currency_m,
               t.last_purchase_ts_m, e.last_engagement_ts_m, i.last_interaction_ts_m,
               (t.customer_id IS NOT NULL)::INT AS has_txn_m,
               (e.customer_id IS NOT NULL)::INT AS has_eng_m,
               (i.customer_id IS NOT NULL)::INT AS has_int_m
        FROM spine s
        LEFT JOIN m_txn t USING (customer_id, month)
        LEFT JOIN m_eng e USING (customer_id, month)
        LEFT JOIN m_int i USING (customer_id, month)
    """)

    # rolling 3-month sums for every monthly count/amount
    roll = [c for cols in m_cols.values() for c in cols if c.endswith("_m")]
    roll_sql = ",\n".join(f"sum({c}) OVER w3 AS {c[:-2]}_3m" for c in roll)
    prev_sql = ",\n".join(f"sum({c}) OVER wp3 AS {c[:-2]}_prev3m"
                          for c in ("purchase_amount_m", "purchase_count_m", "active_days_m",
                                    "eng_event_count_m", "interaction_count_m", "complaint_count_m"))
    lb = ELIGIBILITY_LOOKBACK_MONTHS - 1          # w3 / is_eligible window = trailing N months incl. current

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE cm_roll AS
        WITH a AS (
            SELECT *, (has_txn_m + has_eng_m + has_int_m > 0)::INT AS active_m FROM cm_base
        ), f AS (
            SELECT *, min(month_start) FILTER (WHERE active_m = 1)
                          OVER (PARTITION BY customer_id) AS first_active_month
            FROM a
        )
        SELECT *,
               {roll_sql},
               {prev_sql},
               row_number() OVER wcum                                 AS months_of_history,
               CASE WHEN first_active_month IS NOT NULL AND month_start >= first_active_month
                    THEN date_diff('month', first_active_month, month_start) + 1 ELSE 0 END
                                                                      AS months_observed,
               sum(active_m) OVER wcum                                AS active_months_to_date,
               max(last_purchase_ts_m)    OVER wcum                   AS last_purchase_ts,
               max(last_engagement_ts_m)  OVER wcum                   AS last_engagement_ts,
               max(last_interaction_ts_m) OVER wcum                   AS last_interaction_ts,
               sum(opt_out_count_m)       OVER wcum                   AS opt_out_count_to_date,
               sum(has_txn_m + has_eng_m + has_int_m) OVER (PARTITION BY customer_id ORDER BY month_start
                    ROWS BETWEEN {lb} PRECEDING AND CURRENT ROW)       AS source_months_lookback,
               max(has_txn_m) OVER w3                                 AS has_txn_3m,
               max(has_eng_m) OVER w3                                 AS has_eng_3m,
               max(has_int_m) OVER w3                                 AS has_int_3m
        FROM f
        WINDOW w3   AS (PARTITION BY customer_id ORDER BY month_start ROWS BETWEEN {lb} PRECEDING AND CURRENT ROW),
               wp3  AS (PARTITION BY customer_id ORDER BY month_start
                        ROWS BETWEEN {2 * lb + 1} PRECEDING AND {lb + 1} PRECEDING),
               wcum AS (PARTITION BY customer_id ORDER BY month_start ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
    """)

## 6. Final feature table

In [ ]:
def final_features(con) -> None:
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE cm_features AS
        SELECT
            r.customer_id, r.month, r.month_start, r.month_end,
            (r.source_months_lookback > 0)                                    AS is_eligible,
            r.months_of_history, r.months_observed, r.active_m, r.active_months_to_date,
            r.pre_signup_m,                                                     -- D19: activity before signup_date (DQ flag)

            -- ===== TRANSACTIONS (behavioural health; amounts in native currency) =====
            r.txn_currency_m, r.currency_count_m,
            r.txn_count_m, r.txn_success_count_m, r.txn_failed_count_m,
            r.purchase_count_m, r.purchase_amount_m, r.refund_count_m,
            r.refund_amount_m, r.payment_attempts_m, r.failed_payment_count_m, r.reversal_count_m,
            r.discount_amount_m, r.intl_txn_count_m, r.distinct_categories_m,
            r.fee_count_m, r.fee_amount_m, r.fee_reversed_amount_m,
            r.fee_amount_m - r.fee_reversed_amount_m                            AS net_fee_amount_m,
            r.purchase_count_3m, r.purchase_amount_3m, r.refund_amount_3m,
            r.failed_payment_count_3m, r.payment_attempts_3m, r.reversal_count_3m,
            r.txn_count_3m, r.txn_success_count_3m,
            CASE WHEN r.has_txn_m = 1
                 THEN percent_rank() OVER (PARTITION BY r.month, r.has_txn_m, r.txn_currency_m
                                           ORDER BY r.purchase_amount_m) END  AS spend_pctile_m,
            r.purchase_amount_m / NULLIF(r.purchase_count_m, 0)                AS avg_ticket_m,
            r.txn_success_count_3m / NULLIF(r.txn_count_3m, 0)                 AS txn_success_rate_3m,
            least(r.refund_amount_3m / NULLIF(r.purchase_amount_3m, 0), 1.0)    AS refund_ratio_3m,
            r.failed_payment_count_3m / NULLIF(r.payment_attempts_3m, 0)        AS failed_payment_rate_3m,
            r.discount_amount_3m / NULLIF(r.purchase_amount_3m + r.discount_amount_3m, 0) AS discount_share_3m,
            CASE WHEN r.months_of_history >= {2 * ELIGIBILITY_LOOKBACK_MONTHS}
                 THEN (r.purchase_amount_3m - r.purchase_amount_prev3m)
                      / NULLIF(abs(r.purchase_amount_prev3m), 0) END           AS spend_trend_3m_vs_prev3m,
            CASE WHEN r.months_of_history >= {2 * ELIGIBILITY_LOOKBACK_MONTHS}
                 THEN (r.purchase_count_3m - r.purchase_count_prev3m)
                      / NULLIF(r.purchase_count_prev3m, 0) END                 AS purchase_count_trend_3m_vs_prev3m,
            date_diff('day', CAST(r.last_purchase_ts AS DATE), r.month_end)     AS days_since_last_purchase,

            -- ===== ENGAGEMENTS (behavioural health) =====
            r.eng_event_count_m, r.active_days_m, r.login_count_m, r.session_minutes_m,
            r.pages_viewed_m, r.conversion_count_m, r.opt_out_count_m, r.dismiss_count_m,
            r.eng_positive_count_m, r.eng_neutral_count_m, r.eng_negative_count_m,
            r.email_open_count_m, r.email_click_count_m, r.campaign_touch_count_m, r.eng_channel_count_m,
            r.active_days_3m, r.eng_event_count_3m, r.session_minutes_3m, r.conversion_count_3m,
            r.opt_out_count_3m, r.dismiss_count_3m, r.eng_positive_count_3m, r.eng_negative_count_3m,
            r.opt_out_count_to_date,
            (r.opt_out_count_to_date > 0)                                       AS has_opted_out_to_date,
            r.pages_viewed_m / NULLIF(r.eng_event_count_m, 0)                   AS pages_per_event_m,
            r.session_minutes_m / NULLIF(r.eng_event_count_m, 0)                AS minutes_per_event_m,
            r.eng_positive_count_3m / NULLIF(r.eng_positive_count_3m + r.eng_negative_count_3m, 0)
                                                                                AS positive_engagement_share_3m,
            least(r.email_click_count_3m / NULLIF(r.email_open_count_3m, 0), 1.0) AS email_click_to_open_3m,
            least(r.conversion_count_3m / NULLIF(r.campaign_touch_count_3m, 0), 1.0) AS campaign_conversion_rate_3m,
            CASE WHEN r.months_of_history >= {2 * ELIGIBILITY_LOOKBACK_MONTHS}
                 THEN (r.active_days_3m - r.active_days_prev3m)
                      / NULLIF(r.active_days_prev3m, 0) END                     AS active_days_trend_3m_vs_prev3m,
            date_diff('day', CAST(r.last_engagement_ts AS DATE), r.month_end)   AS days_since_last_engagement,

            -- ===== INTERACTIONS (service signals; no labels) =====
            r.interaction_count_m, r.complaint_count_m, r.billing_dispute_count_m,
            r.cancellation_request_count_m, r.open_issue_count_m, r.escalated_count_m,
            r.repeat_contact_count_m, r.handle_minutes_m, r.int_channel_count_m,
            r.interaction_count_3m, r.complaint_count_3m, r.billing_dispute_count_3m,
            r.cancellation_request_count_3m, r.open_issue_count_3m,
            r.escalated_count_3m, r.repeat_contact_count_3m,
            r.complaint_count_3m / NULLIF(r.interaction_count_3m, 0)            AS complaint_share_3m,
            r.open_issue_count_3m / NULLIF(r.interaction_count_3m, 0)           AS open_issue_share_3m,
            r.repeat_contact_count_3m / NULLIF(r.interaction_count_3m, 0)       AS repeat_contact_share_3m,
            r.fcr_yes_3m / NULLIF(r.fcr_known_3m, 0)                            AS fcr_rate_3m,
            CASE WHEN r.months_of_history >= {2 * ELIGIBILITY_LOOKBACK_MONTHS}
                 THEN r.complaint_count_3m - r.complaint_count_prev3m END       AS complaint_change_3m_vs_prev3m,
            date_diff('day', CAST(r.last_interaction_ts AS DATE), r.month_end)  AS days_since_last_interaction,

            -- ===== CSAT (service validation signal - NULL when not observed, never imputed) =====
            r.csat_n_m, r.csat_n_3m,
            r.csat_sum_m / NULLIF(r.csat_n_m, 0)                                AS csat_avg_m,
            r.csat_sum_3m / NULLIF(r.csat_n_3m, 0)                              AS csat_avg_3m,
            ((r.csat_sum_3m / NULLIF(r.csat_n_3m, 0)) - {br.CSAT_MIN})
                / {br.CSAT_MAX - br.CSAT_MIN}.0 * 100                           AS csat_health_3m,
            r.csat_low_3m, r.csat_high_3m,

            -- ===== NPS (loyalty / independent validation - NOT a BHS or sentiment input) =====
            r.nps_n_m, r.nps_n_3m,
            r.nps_sum_3m / NULLIF(r.nps_n_3m, 0)                                AS nps_avg_3m,
            r.nps_detractor_3m, r.nps_passive_3m, r.nps_promoter_3m,
            100.0 * (r.nps_promoter_3m - r.nps_detractor_3m) / NULLIF(r.nps_n_3m, 0) AS nps_net_3m,

            -- ===== DATA COVERAGE (feeds the confidence band) =====
            r.has_txn_m, r.has_eng_m, r.has_int_m,
            r.has_txn_3m, r.has_eng_3m, r.has_int_3m,
            (r.has_txn_m + r.has_eng_m + r.has_int_m)                           AS sources_present_m,
            (r.has_txn_3m + r.has_eng_3m + r.has_int_3m)                        AS sources_present_3m,
            (r.csat_n_3m > 0)                                                   AS has_csat_3m,
            (r.nps_n_3m > 0)                                                    AS has_nps_3m,

            -- ===== PROFILE CONTEXT (segmentation / fairness / consent only) =====
            date_diff('month', c.signup_date, r.month_end)                      AS ctx_tenure_months,
            c.tenure_months AS ctx_tenure_months_snapshot,
            c.age AS ctx_age, c.gender AS ctx_gender, c.region AS ctx_region, c.country AS ctx_country,
            c.customer_segment AS ctx_customer_segment, c.acquisition_channel AS ctx_acquisition_channel,
            c.income_band AS ctx_income_band, c.preferred_channel AS ctx_preferred_channel,
            c.loyalty_tier AS ctx_loyalty_tier,
            c.marketing_consent AS ctx_marketing_consent,

            current_timestamp AS _built_at
        FROM cm_roll r
        JOIN silver_customers c USING (customer_id)
    """)

## 7. Feature QA

In [ ]:
def feature_qa(con) -> list[dict]:
    cols = [r[0] for r in con.execute("DESCRIBE cm_features").fetchall()]
    leak = [c for c in cols if any(lbl in c for lbl in LABEL_COLUMNS + ["interaction_text", "gt_"])]
    ratio_cols = [c for c in cols if c.endswith(("_ratio_3m", "_rate_3m", "_share_3m", "click_to_open_3m",
                                                 "spend_pctile_m"))]
    count_cols = [c for c in cols if c.endswith(("_count_m", "_count_3m", "_days_m", "_days_3m"))]

    checks = [("label / text columns present", len(leak)),
              ("duplicate customer-month",
               scalar(con, "SELECT count(*) - count(DISTINCT (customer_id, month)) FROM cm_features")),
              ("month outside window",
               scalar(con, f"SELECT count(*) FROM cm_features WHERE month_start < DATE '{WINDOW_START}' "
                           f"OR month_start > DATE '{WINDOW_END}'")),
              ("months_observed > months_of_history",
               scalar(con, "SELECT count(*) FROM cm_features WHERE months_observed > months_of_history")),
              ("has_<src>_m = 0 but activity count > 0",
               scalar(con, "SELECT count(*) FROM cm_features WHERE (has_txn_m = 0 AND txn_count_m > 0) "
                           "OR (has_eng_m = 0 AND eng_event_count_m > 0) "
                           "OR (has_int_m = 0 AND interaction_count_m > 0)")),
              ("source customer-months lost from spine (D19)",
               scalar(con, "SELECT (SELECT count(*) FROM m_txn) + (SELECT count(*) FROM m_eng) "
                           "+ (SELECT count(*) FROM m_int) - (SELECT sum(has_txn_m + has_eng_m + has_int_m) "
                           "FROM cm_features)")),
              ("csat_health_3m outside 0-100",
               scalar(con, "SELECT count(*) FROM cm_features WHERE csat_health_3m < 0 OR csat_health_3m > 100"))]
    for c in ratio_cols:
        checks.append((f"{c} in [0,1]", scalar(con, f"SELECT count(*) FROM cm_features WHERE {c} < 0 OR {c} > 1")))
    neg = " OR ".join(f"{c} < 0" for c in count_cols)
    checks.append(("negative counts", scalar(con, f"SELECT count(*) FROM cm_features WHERE {neg}")))
    for c in ("days_since_last_purchase", "days_since_last_engagement", "days_since_last_interaction"):
        checks.append((f"{c} >= 0", scalar(con, f"SELECT count(*) FROM cm_features WHERE {c} < 0")))
    return [{"check": n, "violations": v, "status": "PASS" if v == 0 else "FAIL"} for n, v in checks]

## 8. Main

In [ ]:
def main(strict: bool = True) -> bool:
    FEATURE_DIR.mkdir(parents=True, exist_ok=True)
    OUT.mkdir(parents=True, exist_ok=True)
    con = get_connection()
    try:
        register_layer(con, SILVER_DIR, "silver")
        mapping_issues = check_mappings(con, strict)

        log("Aggregating sources by customer_id + month")
        monthly_transactions(con)
        monthly_engagements(con)
        monthly_interactions(con)
        monthly_labels(con)

        log("Building dense customer-month spine and rolling windows")
        build_customer_month(con)
        final_features(con)

        log("Running feature QA")
        qa = feature_qa(con)

        copy_to_parquet(con, "SELECT * FROM cm_features", FEATURE_FILE)
        copy_to_parquet(con, """SELECT f.customer_id, f.month, l.* EXCLUDE (customer_id, month)
                                FROM cm_features f JOIN m_lbl l USING (customer_id, month)""", LABEL_FILE)

        summary_q = "SELECT * FROM (SUMMARIZE SELECT * EXCLUDE (_built_at) FROM cm_features)"
        copy_to_csv(con, summary_q, OUT / "feature_summary.csv")
        month_q = """SELECT month, count(*) AS customer_months,
                            count(*) FILTER (WHERE is_eligible) AS eligible,
                            count(*) FILTER (WHERE active_m = 1) AS active,
                            round(avg(sources_present_m), 2) AS avg_sources,
                            round(avg(purchase_amount_m), 2) AS avg_spend_native,
                            round(avg(active_days_m), 2) AS avg_active_days,
                            sum(complaint_count_m) AS complaints
                     FROM cm_features GROUP BY month ORDER BY month"""
        print("\nCUSTOMER-MONTH VOLUME")
        con.sql(month_q).show(max_rows=20)
        copy_to_csv(con, month_q, OUT / "feature_by_month.csv")

        rows, eligible, n_cols = con.execute(
            "SELECT count(*), count(*) FILTER (WHERE is_eligible), "
            "(SELECT count(*) FROM (DESCRIBE cm_features)) FROM cm_features").fetchall()[0]
    finally:
        con.close()

    print("=" * 80)
    print(f"Rows: {rows:,} customer-months | eligible: {eligible:,} | columns: {n_cols}")
    for c in qa:
        print(f"  [{c['status']}] {c['check']:<45} violations = {c['violations']:,}")
    ok = all(c["status"] == "PASS" for c in qa)
    print("=" * 80 + f"\nSTEP 3c STATUS: {'PASSED' if ok else 'FAILED - review checks'}")
    log(f"Features written to {FEATURE_FILE}")
    log(f"Evaluation labels written to {LABEL_FILE} (never join these into model inputs)")
    (OUT / f"feature_audit_{datetime.now():%Y%m%d_%H%M%S}.json").write_text(json.dumps(
        {"run_at": datetime.now().isoformat(timespec="seconds"), "rows": rows,
         "eligible_rows": eligible, "columns": n_cols, "mapping_issues": mapping_issues, "qa": qa,
         "status": "PASSED" if ok else "FAILED"}, indent=2))
    return ok

In [ ]:
if __name__ == "__main__":
    finish(main(strict="--lenient" not in sys.argv))